# Experiment 2: What Happens with Corrupted Shares?

A single corrupted signature share silently produces an invalid group signature.
Can share verification catch it? What about different kinds of corruption?


In [ ]:
import sys
sys.path.insert(0, "../src")

from frost import Participant, Aggregator, Scalar, Point, G, Q
from frost.tagged_hash import tagged_hash

# DKG setup (2-of-3)
threshold = 2
n = 3
participants = [Participant(index=i, threshold=threshold, participants=n) for i in range(1, n + 1)]

for p in participants:
    p.init_keygen()
for p in participants:
    p.generate_shares()
for receiver in participants:
    other_shares = tuple(
        sender.shares[receiver.index - 1]
        for sender in participants if sender.index != receiver.index
    )
    receiver.aggregate_shares(other_shares)
for p in participants:
    other_commitments = tuple(
        other.coefficient_commitments[0]
        for other in participants if other.index != p.index
    )
    p.derive_public_key(other_commitments)
for p in participants:
    other_ccs = tuple(
        other.coefficient_commitments
        for other in participants if other.index != p.index
    )
    p.derive_group_commitments(other_ccs)

public_key = participants[0].public_key

# Signing ceremony (P1 and P2)
signers = [participants[0], participants[1]]
signer_indexes = tuple(p.index for p in signers)
message = b"Honest message"

for signer in signers:
    signer.generate_nonce_pair()

all_nonce_pairs = [(Point(), Point())] * n
for signer in signers:
    all_nonce_pairs[signer.index - 1] = signer.nonce_commitment_pair
nonce_commitment_pairs = tuple(all_nonce_pairs)

signature_shares = []
for signer in signers:
    z_i = signer.sign(message, nonce_commitment_pairs, signer_indexes)
    signature_shares.append(z_i)

print(f"P1 share: {signature_shares[0]}")
print(f"P2 share: {signature_shares[1]}")


## Baseline: Valid Signature

First, confirm a valid signing produces a valid BIP340 signature.


In [ ]:
agg = Aggregator(
    public_key, message, nonce_commitment_pairs, signer_indexes,
    group_commitments=participants[0].group_commitments,
)
good_sig = agg.signature(tuple(signature_shares))

# BIP340 verification (must use even-y public key per lift_x convention)
sig_bytes = bytes.fromhex(good_sig)
R_x = int.from_bytes(sig_bytes[:32], "big")
s = int.from_bytes(sig_bytes[32:], "big")
R = Point.lift_x(R_x)
pk = Point.lift_x(public_key.x)
e_bytes = tagged_hash("BIP0340/challenge", sig_bytes[:32] + pk.to_bytes_xonly() + message)
e = int.from_bytes(e_bytes, "big") % Q
print(f"Valid signature: {s * G == R + (e * pk)}")

## Corruption Without Verification

Corrupt P1's share by adding 1. Aggregate WITHOUT share verification
(no `group_commitments`). The aggregation succeeds, but the signature is invalid.


In [ ]:
corrupted = (signature_shares[0] + 1, signature_shares[1])

agg_no_verify = Aggregator(
    public_key, message, nonce_commitment_pairs, signer_indexes,
)
bad_sig = agg_no_verify.signature(corrupted)

# BIP340 verification (must use even-y public key per lift_x convention)
sig_bytes = bytes.fromhex(bad_sig)
R_x = int.from_bytes(sig_bytes[:32], "big")
s = int.from_bytes(sig_bytes[32:], "big")
R = Point.lift_x(R_x)
pk = Point.lift_x(public_key.x)
e_bytes = tagged_hash("BIP0340/challenge", sig_bytes[:32] + pk.to_bytes_xonly() + message)
e = int.from_bytes(e_bytes, "big") % Q
print(f"Corrupted signature valid? {s * G == R + (e * pk)}")
print("The aggregator accepted the bad share silently.")

## Corruption With Verification

Now aggregate WITH share verification (`group_commitments` provided). The
Aggregator checks each share against the group polynomial before aggregating.


In [ ]:
agg_verify = Aggregator(
    public_key, message, nonce_commitment_pairs, signer_indexes,
    group_commitments=participants[0].group_commitments,
)
try:
    agg_verify.signature(corrupted)
    print("ERROR: should have raised ValueError")
except ValueError as exc:
    print(f"Caught: {exc}")


## Different Corruption Types

Does verification catch every kind of corruption?


In [ ]:
corruptions = {
    "add 1": (signature_shares[0] + 1, signature_shares[1]),
    "multiply by 2": (signature_shares[0] * 2, signature_shares[1]),
    "set to 0": (0, signature_shares[1]),
    "negate": (-signature_shares[0], signature_shares[1]),
}

for name, shares in corruptions.items():
    # Convert to tuple of ints for the aggregator
    int_shares = tuple(s if isinstance(s, int) else int(s) for s in shares)
    agg = Aggregator(
        public_key, message, nonce_commitment_pairs, signer_indexes,
        group_commitments=participants[0].group_commitments,
    )
    try:
        agg.signature(int_shares)
        print(f"  {name}: NOT CAUGHT")
    except ValueError as exc:
        print(f"  {name}: caught ({exc})")


## Two Corrupted Shares

What if both signers are dishonest? With share verification, the first bad
share is caught before the second is checked.


In [ ]:
both_bad = (signature_shares[0] + 1, signature_shares[1] + 1)

agg = Aggregator(
    public_key, message, nonce_commitment_pairs, signer_indexes,
    group_commitments=participants[0].group_commitments,
)
try:
    agg.signature(both_bad)
    print("ERROR: should have raised ValueError")
except ValueError as exc:
    print(f"Caught first bad signer: {exc}")
    print("The second corrupted share was never checked.")


## Results

| Scenario | Verification OFF | Verification ON |
|----------|-----------------|------------------|
| 1 corrupted share | Invalid signature (silent) | ValueError identifies bad signer |
| Different corruption types | All produce invalid signatures | All caught |
| 2 corrupted shares | Invalid signature (silent) | First bad signer identified |

**Key insight:** Without share verification, a single bad signer silently
produces an invalid group signature that looks like a valid 64-byte blob but
fails BIP340 verification. With verification, the bad signer is identified
*before* aggregation, so the coordinator can exclude them or abort.

## Next Steps

- This experiment verifies the same invariant as
  `test_properties.py::test_corrupted_share_detected`.
- What if the corruption is applied to the *nonce commitment* instead of the
  signature share? (The binding value would change, producing a different kind
  of invalid share.)
- See Guide Chapter 7 (Signing) for the verification equation.
